# Nowcasting Meteosat imagery over Ireland with a fine-tuned transformer

This project forecasts what the new **Meteosat Third Generation** satellite will see over Ireland one
and two hours from now.

It starts from NVIDIA's [StormScope-Meteosat](https://huggingface.co/nvidia/stormscope-meteosat), a
diffusion **transformer** pre-trained to predict the next satellite image 10 minutes ahead over
Europe. We keep its 613 million weights frozen and train small **LoRA** adapters (about 0.1% of the
weights) so that it works on **hourly time series** of images over Ireland. The forecasts are then
compared with two baselines: **persistence** (the next hour looks like the last one) and the
**untouched pre-trained model**.

All the code lives in the `earth_forecasting` package
([GitHub](https://github.com/mich3leRusso/earth-forecasting)); this notebook only calls it and shows
the results. The trained adapter is on the Hugging Face Hub at
[`michele007rok/StormScope_Ireland`](https://huggingface.co/michele007rok/StormScope_Ireland).

In [ ]:
import numpy as np
import torch
from physicsnemo.experimental.peft import is_lora_layer

from earth_forecasting import plots
from earth_forecasting.config import load_config
from earth_forecasting.data import FrameWindows
from earth_forecasting.evaluate import finetuned_model

train_cfg = load_config("train.yaml", args=[])  # how the adapter was trained
cfg = load_config("evaluate.yaml", args=[])     # how it is tested: unseen dates, adapter from the Hub
step = np.timedelta64(cfg.data.step_minutes, "m")

## 1. Data: hourly satellite images

The data are **Level 1c images from the Flexible Combined Imager (FCI)** on Meteosat-12 (MTG-I1),
downloaded from the [EUMETSAT Data Store](https://data.eumetsat.int). Each image has **16 channels**:
5 visible, 3 near-infrared, 2 water-vapour and 6 thermal-infrared bands. That lets the model see
clouds in daylight and their temperature day and night.

We crop the full disk to Ireland, which gives 128 × 224 pixels at the satellite's 2 km grid (about
3–4 km on the ground at this latitude), and keep one image per hour.

**One sample** is a short time series: six consecutive hourly images (the last five hours and now)
plus the image one hour later, which is the target. The periods are split in time, never mixed:

| | Forecast start times | Purpose |
|---|---|---|
| Training | 1 June – 31 August 2026 | fit the adapters |
| Validation | 2 – 8 September 2026 | choose the best epoch |
| Test | 1 – 7 October 2026 | final scores, seen by nothing else |

NVIDIA trained the base model on November 2024 – May 2026, so every date here is new to it.
A handful of hours are missing or incomplete on EUMETSAT's side, and the windows that touch them are
skipped.

In [ ]:
for name, c, split in [("training", train_cfg, "train"), ("validation", train_cfg, "val"), ("test", cfg, "test")]:
    print(f"{name:<11} {len(FrameWindows(*c.data[split], c.data.dir, step)):5d} windows")

test = FrameWindows(*cfg.data.test, cfg.data.dir, step, lead_steps=cfg.eval.lead_steps)
model, adapter = finetuned_model(cfg)
example = test[len(test) // 2]  # six context frames + the two observed hours after them
plots.window_figure(model, *example);

## 2. Model: a pre-trained diffusion transformer, fine-tuned with LoRA

Each of StormScope's two "experts" is a **Diffusion Transformer (DiT)**: the image is cut into
4 × 4-pixel patches, which become tokens, and 16 transformer blocks process them with
neighborhood attention and MLPs. A forecast starts from noise and is refined over 48 denoising
steps, each conditioned on the six past images, the sun angles and the terrain. One expert handles
the noisy early steps (large-scale structure), the other the fine detail at the end.

The original model predicts 10 minutes ahead. We switch it to **one-hour steps**: it reads six
hourly images and predicts the next hour. To go further, it feeds its own prediction back in.
The pre-trained weights stay frozen; we train **LoRA adapters** (rank 8) on the attention
projections of the last 8 blocks of both experts, plus the final layer that turns tokens back into
the 16 channels.

In [ ]:
lora = sum(p.numel() for m in model.modules() if is_lora_layer(m) for n, p in m.named_parameters(recurse=False) if "lora" in n)
total = sum(p.numel() for p in model.parameters())
print(f"adapter: {adapter}")
print(f"trained: {lora:,} of {total:,} weights ({lora / total:.2%})")

## 3. Training

Training runs on a laptop GPU with 8 GB of memory, so the batch size is 1, with gradients
accumulated over 8 samples. Each sample trains one expert on the standard diffusion
(denoising) loss. We use the Muon optimizer with a linear warmup and a cosine learning-rate decay,
for 12 epochs over the 2,185 training windows: about 14 minutes per epoch.

Epoch 0 is the untouched pre-trained model on the validation week, which gives the starting point.

In [ ]:
curves = plots.training_curves(train_cfg.output.tensorboard)
ax = curves.plot(marker="o", xlabel="epoch", ylabel="denoising loss", title="Training and validation loss", grid=True)
curves.round(4)

## 4. Evaluation on unseen dates (October 2026)

For 20 forecast start times spread over the test week, each model forecasts one and two hours
ahead, and every forecast is compared with what the satellite actually observed. The score is the
**RMSE** over all land and sea pixels, computed per channel on the normalized scale so that the 16
channels are comparable. Lower is better.

Both models use the same diffusion noise for each date, so the difference between them comes from
the adapters alone. The last two columns show how much each model improves on persistence.

In [ ]:
mean, per_channel = plots.rmse_summary(f"{cfg.output.tensorboard}/rmse_test.csv")
mean[plots.MODELS].plot.bar(rot=0, xlabel="lead time (hours)", ylabel="mean RMSE (normalized)", title="Forecast error on the test week")
mean

In [ ]:
per_channel.round(3)

## 5. One forecast, side by side

The same start time as the example above, for a thermal-infrared channel (cloud-top temperature,
day and night) and a visible one (reflected sunlight). Each row is a lead time. From left to right:
persistence, the pre-trained model, the fine-tuned model, and what the satellite actually saw.

In [ ]:
plots.forecast_figure(model, *example, lead_steps=cfg.eval.lead_steps, channel="fci105ir");

In [ ]:
plots.forecast_figure(model, *example, lead_steps=cfg.eval.lead_steps, channel="fci06vis");

## 6. What we learned

_To be written once the October test results are in._